# Phase 2 – Analyse sémantique des offres d'emploi (AnalyzerAgent)

Ce notebook implémente l'**AnalyzerAgent** du système RecruitAI.
Son rôle est de transformer les offres d'emploi collectées sous forme de texte brut
en représentations structurées exploitables pour les phases de matching et de génération.

L'analyse repose sur le modèle **Gemini** (Google Generative AI).


In [21]:
!pip install -U google-genai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 703.4/703.4 kB 44.6 MB/s eta 0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.54.0
    Uninstalling google-genai-1.54.0:
      Successfully uninstalled google-genai-1.54.0


In [1]:
from google import genai
import os
import json
import re
import time
from typing import Dict, Any, List


In [2]:
from google.colab import userdata

# Chargement sécurisé de la clé API
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

assert os.environ["GEMINI_API_KEY"] is not None, "❌ GEMINI_API_KEY manquante"

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("✅ Gemini API configurée")


✅ Gemini API configurée


##Chargement des offres collectées

In [3]:
with open("job_offers_raw.json", "r", encoding="utf-8") as f:
    job_offers = json.load(f)

print(f"📄 {len(job_offers)} offres chargées")
job_offers[0]


📄 20 offres chargées


{'source': 'Indeed (Google Jobs)',
 'title': 'Emplois : Data Scientist Biology, Paris (75) - 13 décembre 2025 ...',
 'company': 'fr.indeed.com',
 'location': 'Paris',
 'description': "Vous êtes à la recherche d'un emploi : Data Scientist Biology ? Il y en a 22 disponibles pour Paris (75) sur Indeed.com, le plus grand site d'emploi\xa0...",
 'url': 'https://fr.indeed.com/q-data-scientist-biology-l-paris-(75)-emplois.html'}

##Prompt d’analyse

In [9]:
ANALYSIS_PROMPT = """
Tu es un expert en recrutement.

Analyse l'offre d'emploi suivante.
Même si certaines informations ne sont pas explicitement mentionnées,
déduis-les raisonnablement à partir du contexte du poste.

Retourne STRICTEMENT un JSON valide avec les champs :

- title
- company
- skills (liste de compétences techniques probables)
- soft_skills (liste de soft skills probables)
- missions (liste de missions probables)
- experience_level
- contract_type
- seniority

Offre :
\"\"\"
{job_text}
\"\"\"
"""


##AnalyzerAgent

In [12]:
def analyze_job_offer(job: Dict[str, Any]) -> Dict[str, Any]:
    job_text = f"""
Titre: {job.get('title', '')}
Entreprise: {job.get('company', '')}
Description: {job.get('description', '')}
"""

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=ANALYSIS_PROMPT.format(job_text=job_text)
    )

    try:
        json_text = re.search(r"\{.*\}", response.text, re.S).group()
        analysis = json.loads(json_text)
    except Exception:
        analysis = {
            "title": job.get("title"),
            "company": job.get("company"),
            "skills": [],
            "soft_skills": [],
            "missions": [],
            "experience_level": "Unknown",
            "contract_type": "Unknown",
            "seniority": "Unknown"
        }

    return analysis


In [13]:
test_analysis = analyze_job_offer(job_offers[0])
test_analysis


{'title': 'Data Scientist Biology',
 'company': "Indeed.com (déduit de la source de l'offre)",
 'skills': ['Data Science',
  'Biology',
  'Python (probable, standard en Data Science)',
  'R (probable, standard en Data Science)',
  'Statistical Modeling',
  'Machine Learning',
  'Data Visualization',
  'Bioinformatics (probable)',
  'Genomics (possible, selon la spécialisation en Biology)',
  'Data Analysis',
  'Data Mining'],
 'soft_skills': ['Problem-solving',
  'Communication (verbal and written)',
  'Critical Thinking',
  'Teamwork',
  'Adaptability',
  'Curiosity',
  'Presentation skills (to present findings)',
  'Project management (potentially)'],
 'missions': ['Analyzing biological datasets',
  'Developing and implementing machine learning models for biological applications',
  'Performing statistical analysis on biological data',
  'Communicating results to stakeholders',
  'Collaborating with biologists and other data scientists',
  'Data cleaning and preprocessing',
  'Develo

##Analyse batch des offres

In [15]:
analyzed_offers = []

for i, job in enumerate(job_offers):
    print(f" Analyse de l'offre {i+1}/{len(job_offers)}")

    analysis = analyze_job_offer(job)
    analysis["source_url"] = job.get("url")

    analyzed_offers.append(analysis)
    time.sleep(1)  # protection quota


 Analyse de l'offre 1/20
 Analyse de l'offre 2/20
 Analyse de l'offre 3/20
 Analyse de l'offre 4/20
 Analyse de l'offre 5/20
 Analyse de l'offre 6/20
 Analyse de l'offre 7/20
 Analyse de l'offre 8/20
 Analyse de l'offre 9/20
 Analyse de l'offre 10/20
 Analyse de l'offre 11/20
 Analyse de l'offre 12/20
 Analyse de l'offre 13/20
 Analyse de l'offre 14/20
 Analyse de l'offre 15/20
 Analyse de l'offre 16/20
 Analyse de l'offre 17/20
 Analyse de l'offre 18/20
 Analyse de l'offre 19/20
 Analyse de l'offre 20/20


In [16]:
analyzed_offers[:2]


[{'title': 'Data Scientist Biology',
  'company': 'Indeed.com (probablement annonce relayée)',
  'skills': ['Data Science',
   'Biology',
   'Python',
   'R',
   'Statistical Modeling',
   'Machine Learning',
   'Data Analysis',
   'Data Visualization',
   'Bioinformatics',
   'Genomics',
   'Drug Discovery (si pertinent)',
   'Clinical Data Analysis (si pertinent)',
   'SQL',
   'Data Mining',
   'Experiment Design'],
  'soft_skills': ['Problem-solving',
   'Communication (written and verbal)',
   'Teamwork',
   'Critical thinking',
   'Analytical skills',
   'Presentation skills',
   'Project Management',
   'Adaptability',
   'Curiosity',
   'Attention to detail'],
  'missions': ['Analyze biological datasets to identify trends and patterns.',
   'Develop and implement machine learning models for biological applications.',
   'Collaborate with biologists and other scientists to define research questions and develop data-driven solutions.',
   'Communicate findings to both technical a

In [17]:
with open("job_offers_analyzed.json", "w", encoding="utf-8") as f:
    json.dump(analyzed_offers, f, indent=2, ensure_ascii=False)

print("job_offers_analyzed.json généré")


job_offers_analyzed.json généré
